# P01 · Radiografía de un mercado financiero
**QuantLab | Módulo 1 · Fundamentos financieros**

## 1. Objetivo
Comprobar que nuestro entorno funciona y familiarizarnos con el Bid, Ask y último precio de BTC/EUR. Este cuaderno es el punto de partida, no una estrategia de inversión.

## 2. Configuración
En Colab selecciona CPU. En local utiliza el entorno `.venv` y el kernel QuantLab. Ejecuta todas las celdas en orden. Si faltan dependencias en Colab, ejecuta `%pip install requests pandas numpy` en una celda aparte.

In [1]:
import sys
import platform
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

print("Python:", sys.version.split()[0])
print("Sistema:", platform.system())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

Python: 3.13.5
Sistema: Linux
Pandas: 2.2.3
NumPy: 2.3.5
Requests: 2.32.5


## 3. Datos y procedencia
Por defecto utilizamos un ejemplo **ficticio**, de modo que la práctica es reproducible sin conexión. Cuando quieras consultar datos reales, cambia `USE_LIVE_DATA` a `True` y vuelve a ejecutar la celda. La hora de recepción no es la hora del evento del exchange.

Fuente para el modo real: [documentación del ticker público de Kraken](https://docs.kraken.com/api/docs/rest-api/get-ticker-information/).

In [2]:
USE_LIVE_DATA = False

if USE_LIVE_DATA:
    response = requests.get(
        "https://api.kraken.com/0/public/Ticker",
        params={"pair": "XBTEUR"}, timeout=15
    )
    response.raise_for_status()
    payload = response.json()
    if payload["error"]:
        raise RuntimeError(payload["error"])
    ticker = next(iter(payload["result"].values()))
    market = {
        "source": "kraken_public_ticker",
        "received_at": datetime.now(timezone.utc).isoformat(),
        "symbol": "BTC/EUR",
        "best_bid": float(ticker["b"][0]),
        "best_ask": float(ticker["a"][0]),
        "last_price": float(ticker["c"][0]),
    }
else:
    market = {
        "source": "synthetic_example",
        "received_at": None,
        "symbol": "BTC/EUR",
        "best_bid": 59998.0,
        "best_ask": 60002.0,
        "last_price": 59999.0,
    }
pd.DataFrame([market]).T.rename(columns={0: "Valor"})

,Valor
source,synthetic_example
received_at,None
symbol,BTC/EUR
best_bid,59998.0
best_ask,60002.0
last_price,59999.0


## 4. Interpretación financiera
- **Bid:** mejor precio comprador disponible.
- **Ask:** mejor precio vendedor disponible.
- **Last price:** precio de la última operación ejecutada; no tiene por qué coincidir con la cotización actual.

Una compra inmediata de pequeño tamaño cruza el Ask y una venta inmediata cruza el Bid, si hay profundidad suficiente.

In [3]:
assert market["best_bid"] > 0
assert market["best_ask"] >= market["best_bid"]
assert market["last_price"] > 0
print("Comprobaciones básicas superadas.")

Comprobaciones básicas superadas.


## 5. Ejercicio pendiente
Implementa una función que calcule `mid_price`, `spread`, `relative_spread`, `spread_bps` y `last_price_deviation`. Usa las lecciones 1.1 y 1.2 como referencia. Con el ejemplo ficticio, comprueba que el mid es 60.000 EUR y el spread 4 EUR.

¿Qué precio utilizarías para comprar inmediatamente? ¿Y para vender? ¿Cuál sería el coste implícito de una compra y venta de 0,01 BTC, sin comisiones y con precios constantes?

## 6. Conclusiones
Completar tras hacer el ejercicio. Todavía no podemos inferir ninguna señal predictiva a partir de una única observación.